In [3]:
from qdrant_client import QdrantClient
from qdrant_client.http import models
from dotenv import load_dotenv, find_dotenv
import os

load_dotenv(find_dotenv())

# 1. 클라이언트 연결
client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))
COLLECTION_NAME = "attractions_hybrid"
def migrate_payload_to_geo():
    print("🔄 데이터 변환 및 병합 시작...")
    
    offset = None
    processed_count = 0
    pass_count = 0

    while True:
        # 2. 데이터 가져오기
        points, next_offset = client.scroll(
            collection_name=COLLECTION_NAME,
            limit=100,
            offset=offset,
            with_payload=True
        )
        
        if not points:
            break

        # 3. 데이터 변환 및 즉시 업데이트
        # overwrite_payload는 포인트마다 개별적인 페이로드를 가져야 하므로
        # 리스트에 모았다가 한 번에 처리하기보다, 루프 안에서 처리하는 것이 로직상 안전합니다.
        
        for point in points:
            payload = point.payload
            
            # latitude와 longitude가 모두 있는 경우에만 작업 수행
            if "latitude" in payload and "longitude" in payload:
                
                # (1) 기존 페이로드 복사 (title, overview 등 다른 데이터 보존을 위해 필수!)
                new_payload = payload.copy()
                new_payload.pop("location", None)
                # (2) 새로운 location 필드 생성
                new_payload["location"] = {
                    "lat": float(payload["latitude"]),
                    "lon": float(payload["longitude"])
                }
                
                # (3) 기존 필드 삭제 (Python 딕셔너리에서 제거)
                # pop을 사용하면 키가 없어도 에러가 나지 않아 안전합니다.
                new_payload.pop("latitude", None)
                new_payload.pop("longitude", None)
                
                # (4) 덮어쓰기 (Overwrite)
                # set_payload 대신 overwrite_payload를 사용하면
                # 우리가 만든 new_payload로 DB의 메타데이터가 완전히 교체됩니다.
                # (벡터 데이터는 건드리지 않으니 안심하세요)
                client.overwrite_payload(
                    collection_name=COLLECTION_NAME,
                    payload=new_payload,
                    points=[point.id] 
                )
                processed_count += 1
            else:
                pass_count += 1

        print(f"✅ {processed_count}개 누적 처리 완료...")
        print(f"❌ {pass_count}개 누적 건너뛰기...")
        # ★ 무한 루프 방지 로직
        if next_offset is None:
            break
            
        offset = next_offset

    print("🎉 모든 데이터 구조 변경 완료!")

    # 4. Geo Index 생성
    print("⚡ Geo Index 생성 중...")
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name="location",
        field_schema=models.PayloadSchemaType.GEO
    )
    print("✅ Geo Index 생성 완료")

In [ ]:
migrate_payload_to_geo()

🔄 데이터 변환 및 병합 시작...
✅ 0개 누적 처리 완료...
❌ 100개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 200개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 300개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 400개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 500개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 600개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 700개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 800개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 900개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 1000개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 1100개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 1200개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 1300개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 1400개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 1500개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 1600개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 1700개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 1800개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 1900개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 2000개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 2100개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 2200개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 2300개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 2400개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 2500개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 2600개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
❌ 2700개 누적 건너뛰기...
✅ 0개 누적 처리 완료...
